### INDIVIDUAL CURVE CREATOR FOR LINE 6

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

#########################################
''''

Setup - will be replaced once integrated with the front end
Will be using project 5493 bc I like the graphs

I removed the timeline from project 5493 so you can see how the curve is
predicted from just the base features: Gross Sq Footage, Projected Budget,
Projected Commitments, and Estimate at Completion

'''

# import project 5493 (I changed it to some other projects for testing)
filepath = "C4215.csv"
project_code = Path(filepath).stem
df = pd.read_csv(filepath)

print("=== RAW INPUT: ===")
display(df.head(100))

# assign gross sq footage
gross_sq = 100000

#########################################

# Some formatting/ cleaning up stuff
df_cleaned = df.iloc[:, 1:].copy()
if len(df_cleaned) > 0:
    df_cleaned.iat[-1, 0] = "Total"
    start_col_idx = 6
    if df_cleaned.shape[1] > start_col_idx:
        last_row = df_cleaned.iloc[[-1], start_col_idx:]
        cleaned_last_row = (
            last_row.astype("string")
            .apply(lambda s: s.str.split(r"\r\n|\n|\r", regex=True).str[0])
        )
        df_cleaned.iloc[-1, start_col_idx:] = cleaned_last_row.iloc[0].to_numpy()
df_cleaned = df_cleaned.reset_index(drop=True)
df = df_cleaned

# Keep only line 6 values
line6_only = df[df["Line Item"].astype(str).str.lstrip("0").eq("6")].copy()
line6_only = line6_only.drop(columns=["Line Item", "Description"])
df = line6_only.reset_index(drop=True)

# Add project code
df.insert(0, "Project Code", project_code)

# Add gross sq footage
df.insert(1, "Gross Sq Footage", gross_sq)

# Remove "Actuals" columns
cols_to_drop = ["Actuals To Date", "Actuals + Projections"]
df = df.drop(columns=cols_to_drop, errors="ignore")

# Drop all columns after "Estimate at Completion"
_eac = "Estimate at Completion"
if _eac in df.columns:
    _i = list(df.columns).index(_eac)
    df = df.iloc[:, : _i + 1]

# Convert feature values to float
df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']] = (
df[['Gross Sq Footage', 'Projected Budget', 'Projected Commitments', 'Estimate at Completion']]
.replace({'[\$,]': ''}, regex=True)
.astype(float)
)

# Add ratio features
df['EAC_Budget_Ratio'] = (df['Estimate at Completion'] / df['Projected Budget'].replace(0, np.nan))
df['Commitments_Budget_Ratio'] = (df['Projected Commitments'] / df['Projected Budget'].replace(0, np.nan))

# Fill values derived from training set; used as fallback if sq footage is missing
MEAN_SQ_FOOTAGE = 86068.1878
MEDIAN_BUDGET_PER_SQFT = 12.4897
df['Gross Sq Footage'] = df['Gross Sq Footage'].fillna(MEAN_SQ_FOOTAGE)
df['Budget_per_SqFt'] = df['Projected Budget'] / df['Gross Sq Footage'].replace(0, np.nan)
df['Budget_per_SqFt'] = df['Budget_per_SqFt'].fillna(MEDIAN_BUDGET_PER_SQFT)
df['Log_Projected_Budget'] = np.log1p(df['Projected Budget'])
df['Log_Gross_SqFt'] = np.log1p(df['Gross Sq Footage'])

#########################################
''''

IMPORTING THE MODELS

'''
#########################################

import joblib

# Duration
duration_model_path = Path("trained_models/Duration_Months_model_l6_v1.joblib")
duration_model = joblib.load(duration_model_path)

# S_Curve_k
k_model_path = Path("trained_models/S_Curve_k_model_l6_v1.joblib")
k_model = joblib.load(k_model_path)

# S_Curve_t0
t0_model_path = Path("trained_models/S_Curve_t0_model_l6_v1.joblib")
t0_model = joblib.load(t0_model_path)

feature_cols = [
    'Gross Sq Footage', 'Projected Budget', 'Projected Commitments',
    'Estimate at Completion', 'EAC_Budget_Ratio', 'Commitments_Budget_Ratio',
    'Budget_per_SqFt', 'Log_Projected_Budget', 'Log_Gross_SqFt'
]

print("=== PREDICTIONS: ===")

# Predict raw model outputs
duration_pred = duration_model.predict(df[feature_cols])[0]
print("Duration: ", duration_pred)
log_k_pred  = k_model.predict(df[feature_cols])[0]
t0_rel_pred = t0_model.predict(df[feature_cols])[0]

# De-transform: models output log(k) and t0/duration, not raw values
k_pred  = np.exp(log_k_pred)
t0_pred = t0_rel_pred * duration_pred
print("Growth Rate: ", k_pred)
print("Midpoint: ", t0_pred)

L_pred = float(df['Projected Commitments'].values[0])
print("Max Value: ", L_pred)

from matplotlib.ticker import MultipleLocator, FuncFormatter

def logistic_curve(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

### DISPLAY THE CURVE
def format_currency(x, pos):
    if x >= 1e6:
        return f'${x * 1e-6:.1f}M'
    elif x >= 1e3:
        return f'${x * 1e-3:.0f}K'
    else:
        return f'${x:,.0f}'

# Generate X axis and calculate Y values
x_data = np.arange(int(duration_pred))
predicted_y = logistic_curve(x_data, L_pred, k_pred, t0_pred)

# Plot the curve
plt.figure(figsize=(9, 4))
plt.plot(x_data, predicted_y, linewidth=3, color='#2ca02c', label="Parameterized S-Curve")
plt.title(f"Standardized S-Curve (Line 6): Project {project_code}", fontsize=14)
plt.xlabel("Months from Project Start", fontsize=12) 
plt.ylabel("Cumulative Cost", fontsize=12) 
ax = plt.gca() 

# Formatting the graph
ax.xaxis.set_major_locator(MultipleLocator(3))
ax.xaxis.set_minor_locator(MultipleLocator(1))
ax.yaxis.set_major_formatter(FuncFormatter(format_currency))
ax.yaxis.set_major_locator(plt.MaxNLocator(8)) 
ax.yaxis.set_minor_locator(plt.MaxNLocator(16)) 
plt.grid(which='major', color='#CCCCCC', linestyle='-', linewidth=0.8)
plt.grid(which='minor', color='#EEEEEE', linestyle='--', linewidth=0.5)
plt.legend()
plt.tight_layout()
plt.show()